In [4]:
import pandas as pd
import numpy as np

In [5]:
#Bước 1: Đọc dữ liệu
df = pd.read_csv("shipments_realistic.csv",header = 0)
df

,shipper_id,order_id,ship_date,delivery_date,shipping_fee,shipper_company,shipper_vehicle,shipper_experience_years,shipper_rating,delivery_success_rate,...,join_date,shipper_name,shipper_phone,shipper_gender,shipper_age,shipper_marital_status,shipper_education,city,region,district
0,SHP00001,1,7/7/2012,7/11/2012,1.37,Viettel Post,Truck,7,5.0,99.0,...,3/17/2026,Bùi Văn Long,991476209,Male,27,Married,Bachelor,Phan Rang-Thap Cham,Central,District #25
1,SHP00002,2,7/6/2012,7/10/2012,2.60,J&T Express,Van,2,4.9,98.4,...,1/29/2025,Trần Anh Khánh,959297982,Male,41,Married,Bachelor,Phan Thiet,Central,District #29
2,SHP00003,3,7/4/2012,7/7/2012,2.38,GHN,Motorbike,10,4.8,95.1,...,11/13/2019,Hoàng Thị Khánh,927142576,Male,30,Single,High School,Long Xuyen,West,District #34
3,SHP00004,4,7/5/2012,7/11/2012,2.49,Viettel Post,Truck,8,5.0,96.3,...,12/22/2025,Trần Đức Vy,971617475,Female,42,Married,College,Kon Tum,Central,District #27
4,SHP00005,6,7/9/2012,7/16/2012,25.79,BEST Express,Truck,10,4.6,95.7,...,12/19/2019,Trần Minh Cường,979196342,Male,31,Married,College,Da Nang,Central,District #23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
566062,SHP00078,834196,12/29/2022,12/31/2022,2.22,Shopee Express,Motorbike,11,5.0,96.8,...,9/25/2025,Đặng Văn Bình,999125359,Female,29,Single,High School,Hoi An,Central,District #25
566063,SHP00003,834293,12/29/2022,12/31/2022,26.83,GHN,Motorbike,10,4.8,95.1,...,11/13/2019,Hoàng Thị Khánh,927142576,Male,30,Single,High School,Long Xuyen,West,District #34
566064,SHP00012,834299,12/28/2022,12/31/2022,1.23,GHTK,Motorbike,6,4.7,97.2,...,10/1/2023,Nguyễn Thị Bình,972210652,Female,49,Single,Bachelor,Ha Long,East,District #07
566065,SHP00039,834314,12/29/2022,12/31/2022,2.28,Ahamove,Motorbike,8,4.9,97.3,...,8/3/2021,Phạm Đức Giang,954844436,Female,31,Single,High School,Nam Dinh,East,District #08


In [6]:
#Bước 2: Kiểm tra
# Null theo từng cột
print(df.isnull().sum())

shipper_id                  0
order_id                    0
ship_date                   0
delivery_date               0
shipping_fee                0
shipper_company             0
shipper_vehicle             0
shipper_experience_years    0
shipper_rating              0
delivery_success_rate       0
average_delivery_time       0
working_shift               0
join_date                   0
shipper_name                0
shipper_phone               0
shipper_gender              0
shipper_age                 0
shipper_marital_status      0
shipper_education           0
city                        0
region                      0
district                    0
dtype: int64


In [7]:
# Duplicate toàn dòng
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [8]:
# order_id phải là khóa chính của shipment -> không được trùng
print("Duplicate order_id:", df['order_id'].duplicated().sum())

Duplicate order_id: 0


In [9]:
# Kiểm tra dtype
print(df.dtypes)

shipper_id                   object
order_id                      int64
ship_date                    object
delivery_date                object
shipping_fee                float64
shipper_company              object
shipper_vehicle              object
shipper_experience_years      int64
shipper_rating              float64
delivery_success_rate       float64
average_delivery_time         int64
working_shift                object
join_date                    object
shipper_name                 object
shipper_phone                 int64
shipper_gender               object
shipper_age                   int64
shipper_marital_status       object
shipper_education            object
city                         object
region                       object
district                     object
dtype: object


In [10]:
# Bước 3 - Kiểm tra tính nhất quán của SHIPPER
shipper_cols = ['shipper_id','shipper_name','shipper_phone','shipper_company',
                'shipper_vehicle','shipper_experience_years','shipper_rating',
                'delivery_success_rate','average_delivery_time','working_shift',
                'join_date','shipper_gender','shipper_age','shipper_marital_status',
                'shipper_education','city','district','region']

In [11]:
# Với mỗi shipper_id, đếm số giá trị KHÁC NHAU của từng cột shipper
inconsistency = df.groupby('shipper_id')[shipper_cols[1:]].nunique()
# Nếu shipper_id nhất quán, mọi giá trị nunique phải = 1
bad_shippers = inconsistency[(inconsistency > 1).any(axis=1)]
print("Số shipper có dữ liệu KHÔNG nhất quán:", bad_shippers.shape[0])

Số shipper có dữ liệu KHÔNG nhất quán: 0


In [12]:
# Bước 4 - Tách bảng SHIPPER (dedupe theo shipper_id)
silver_shipper = df[shipper_cols].drop_duplicates(subset=['shipper_id']).reset_index(drop=True)
print(silver_shipper.shape)
print("Số shipper_id trùng còn sót:", silver_shipper['shipper_id'].duplicated().sum())

# Sửa shipper_phone: ép về string, thêm lại số 0 đầu cho đủ 10 số
silver_shipper['shipper_phone'] = (
    silver_shipper['shipper_phone']
    .astype('Int64')       # về int trước để loại bỏ .0 nếu có
    .astype(str)
    .str.zfill(10)          # đệm thêm số 0 ở đầu cho đủ 10 ký tự
)

# Kiểm tra lại: toàn bộ phải đúng 10 ký tự
print("Số dòng shipper_phone KHÔNG đủ 10 số:", (silver_shipper['shipper_phone'].str.len() != 10).sum())

silver_shipper

(80, 18)
Số shipper_id trùng còn sót: 0
Số dòng shipper_phone KHÔNG đủ 10 số: 0


,shipper_id,shipper_name,shipper_phone,shipper_company,shipper_vehicle,shipper_experience_years,shipper_rating,delivery_success_rate,average_delivery_time,working_shift,join_date,shipper_gender,shipper_age,shipper_marital_status,shipper_education,city,district,region
0,SHP00001,Bùi Văn Long,0991476209,Viettel Post,Truck,7,5.0,99.0,61,Evening,3/17/2026,Male,27,Married,Bachelor,Phan Rang-Thap Cham,District #25,Central
1,SHP00002,Trần Anh Khánh,0959297982,J&T Express,Van,2,4.9,98.4,72,Afternoon,1/29/2025,Male,41,Married,Bachelor,Phan Thiet,District #29,Central
2,SHP00003,Hoàng Thị Khánh,0927142576,GHN,Motorbike,10,4.8,95.1,53,Evening,11/13/2019,Male,30,Single,High School,Long Xuyen,District #34,West
3,SHP00004,Trần Đức Vy,0971617475,Viettel Post,Truck,8,5.0,96.3,53,Evening,12/22/2025,Female,42,Married,College,Kon Tum,District #27,Central
4,SHP00005,Trần Minh Cường,0979196342,BEST Express,Truck,10,4.6,95.7,62,Morning,12/19/2019,Male,31,Married,College,Da Nang,District #23,Central
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,SHP00076,Trần Thanh Long,0933140008,BEST Express,Truck,10,4.7,95.1,65,Afternoon,9/25/2020,Male,29,Married,Bachelor,Ninh Binh,District #01,East
76,SHP00077,Trần Văn Giang,0931398461,BEST Express,Truck,9,4.7,99.4,41,Morning,11/11/2019,Female,38,Married,Bachelor,Quy Nhon,District #23,Central
77,SHP00078,Đặng Văn Bình,0999125359,Shopee Express,Motorbike,11,5.0,96.8,38,Evening,9/25/2025,Female,29,Single,High School,Hoi An,District #25,Central
78,SHP00079,Đặng Thanh An,0993275153,Shopee Express,Motorbike,11,4.5,95.9,68,Morning,9/9/2018,Male,46,Married,Bachelor,Ho Chi Minh City,District #35,West


In [13]:
# Bước 5: Tách bảng SHIPMENT
shipment_cols = ['order_id','shipper_id','ship_date','delivery_date','shipping_fee']

silver_shipment = df[shipment_cols].copy()

print(silver_shipment.shape)
print("Duplicate order_id:", silver_shipment['order_id'].duplicated().sum())
silver_shipment

(566067, 5)
Duplicate order_id: 0


,order_id,shipper_id,ship_date,delivery_date,shipping_fee
0,1,SHP00001,7/7/2012,7/11/2012,1.37
1,2,SHP00002,7/6/2012,7/10/2012,2.60
2,3,SHP00003,7/4/2012,7/7/2012,2.38
3,4,SHP00004,7/5/2012,7/11/2012,2.49
4,6,SHP00005,7/9/2012,7/16/2012,25.79
...,...,...,...,...,...
566062,834196,SHP00078,12/29/2022,12/31/2022,2.22
566063,834293,SHP00003,12/29/2022,12/31/2022,26.83
566064,834299,SHP00012,12/28/2022,12/31/2022,1.23
566065,834314,SHP00039,12/29/2022,12/31/2022,2.28


In [14]:
#Bước 6 - Chuẩn hóa kiểu dữ liệu (date, numeric)
# Chuyển ngày tháng
#errors='coerce' biến giá trị parse lỗi thành NaT để mình phát hiện được thay vì crash
silver_shipment['ship_date'] = pd.to_datetime(silver_shipment['ship_date'], format='%m/%d/%Y', errors='coerce')
silver_shipment['delivery_date'] = pd.to_datetime(silver_shipment['delivery_date'], format='%m/%d/%Y', errors='coerce')
silver_shipper['join_date'] = pd.to_datetime(silver_shipper['join_date'], format='%m/%d/%Y', errors='coerce')

# Kiểm tra có ngày nào parse lỗi (NaT) không
print("Lỗi ship_date:", silver_shipment['ship_date'].isna().sum())
print("Lỗi delivery_date:", silver_shipment['delivery_date'].isna().sum())
print("Lỗi join_date:", silver_shipper['join_date'].isna().sum())

# Kiểm tra logic: delivery_date phải >= ship_date
invalid_dates = silver_shipment[silver_shipment['delivery_date'] < silver_shipment['ship_date']]
print("Số dòng delivery_date < ship_date:", invalid_dates.shape[0])

Lỗi ship_date: 0
Lỗi delivery_date: 0
Lỗi join_date: 0
Số dòng delivery_date < ship_date: 0


In [15]:
#Bước 7 - Kiểm tra tham chiếu khóa ngoại (FK)
# Mọi shipper_id trong SHIPMENT phải tồn tại trong SHIPPER
valid_shipper_ids = set(silver_shipper['shipper_id'])
orphan_shipments = silver_shipment[~silver_shipment['shipper_id'].isin(valid_shipper_ids)]

print("Số shipment có shipper_id KHÔNG tồn tại trong SHIPPER:", orphan_shipments.shape[0])
orphan_shipments.head()

Số shipment có shipper_id KHÔNG tồn tại trong SHIPPER: 0


,order_id,shipper_id,ship_date,delivery_date,shipping_fee


In [16]:
#Bước 8 - Kiểm tra final trước khi export
print("=== SILVER_SHIPPER ===")
print("Rows:", silver_shipper.shape[0], "| Unique shipper_id:", silver_shipper['shipper_id'].nunique())
print("Null counts:\n", silver_shipper.isnull().sum())

print("\n=== SILVER_SHIPMENT ===")
print("Rows:", silver_shipment.shape[0], "| Unique order_id:", silver_shipment['order_id'].nunique())
print("Null counts:\n", silver_shipment.isnull().sum())

=== SILVER_SHIPPER ===
Rows: 80 | Unique shipper_id: 80
Null counts:
 shipper_id                  0
shipper_name                0
shipper_phone               0
shipper_company             0
shipper_vehicle             0
shipper_experience_years    0
shipper_rating              0
delivery_success_rate       0
average_delivery_time       0
working_shift               0
join_date                   0
shipper_gender              0
shipper_age                 0
shipper_marital_status      0
shipper_education           0
city                        0
district                    0
region                      0
dtype: int64

=== SILVER_SHIPMENT ===
Rows: 566067 | Unique order_id: 566067
Null counts:
 order_id         0
shipper_id       0
ship_date        0
delivery_date    0
shipping_fee     0
dtype: int64


In [17]:
silver_shipper

,shipper_id,shipper_name,shipper_phone,shipper_company,shipper_vehicle,shipper_experience_years,shipper_rating,delivery_success_rate,average_delivery_time,working_shift,join_date,shipper_gender,shipper_age,shipper_marital_status,shipper_education,city,district,region
0,SHP00001,Bùi Văn Long,0991476209,Viettel Post,Truck,7,5.0,99.0,61,Evening,2026-03-17,Male,27,Married,Bachelor,Phan Rang-Thap Cham,District #25,Central
1,SHP00002,Trần Anh Khánh,0959297982,J&T Express,Van,2,4.9,98.4,72,Afternoon,2025-01-29,Male,41,Married,Bachelor,Phan Thiet,District #29,Central
2,SHP00003,Hoàng Thị Khánh,0927142576,GHN,Motorbike,10,4.8,95.1,53,Evening,2019-11-13,Male,30,Single,High School,Long Xuyen,District #34,West
3,SHP00004,Trần Đức Vy,0971617475,Viettel Post,Truck,8,5.0,96.3,53,Evening,2025-12-22,Female,42,Married,College,Kon Tum,District #27,Central
4,SHP00005,Trần Minh Cường,0979196342,BEST Express,Truck,10,4.6,95.7,62,Morning,2019-12-19,Male,31,Married,College,Da Nang,District #23,Central
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,SHP00076,Trần Thanh Long,0933140008,BEST Express,Truck,10,4.7,95.1,65,Afternoon,2020-09-25,Male,29,Married,Bachelor,Ninh Binh,District #01,East
76,SHP00077,Trần Văn Giang,0931398461,BEST Express,Truck,9,4.7,99.4,41,Morning,2019-11-11,Female,38,Married,Bachelor,Quy Nhon,District #23,Central
77,SHP00078,Đặng Văn Bình,0999125359,Shopee Express,Motorbike,11,5.0,96.8,38,Evening,2025-09-25,Female,29,Single,High School,Hoi An,District #25,Central
78,SHP00079,Đặng Thanh An,0993275153,Shopee Express,Motorbike,11,4.5,95.9,68,Morning,2018-09-09,Male,46,Married,Bachelor,Ho Chi Minh City,District #35,West


In [18]:
silver_shipment

,order_id,shipper_id,ship_date,delivery_date,shipping_fee
0,1,SHP00001,2012-07-07,2012-07-11,1.37
1,2,SHP00002,2012-07-06,2012-07-10,2.60
2,3,SHP00003,2012-07-04,2012-07-07,2.38
3,4,SHP00004,2012-07-05,2012-07-11,2.49
4,6,SHP00005,2012-07-09,2012-07-16,25.79
...,...,...,...,...,...
566062,834196,SHP00078,2022-12-29,2022-12-31,2.22
566063,834293,SHP00003,2022-12-29,2022-12-31,26.83
566064,834299,SHP00012,2022-12-28,2022-12-31,1.23
566065,834314,SHP00039,2022-12-29,2022-12-31,2.28


In [19]:
#Bước 9 - Export ra CSV (đã sửa: dùng utf-8-sig để Excel hiển thị đúng tiếng Việt)
silver_shipper.to_csv('silver_shipper.csv', index=False, encoding='utf-8-sig')
silver_shipment.to_csv('silver_shipment.csv', index=False, encoding='utf-8-sig')